# LSTM Next-Word Predictor — Encoder/Decoder with Luong Attention (PyTorch)

Trained on the review text of the **IMDB 50K Movie Reviews** dataset (sentiment column dropped).

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or better).

Pipeline: download via Kaggle API → clean & tokenize → dedupe & split → vocabulary (`<pad> <unk> <start> <end>`) → sliding windows → 12-layer residual LSTM encoder + 12-layer residual LSTM decoder with Luong (general) attention → train → evaluate → predict.

## 1. Environment

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "kaggle"], check=True)

import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Runtime -> Change runtime type -> T4 GPU. A 12-layer model on CPU is impractically slow.")

## 2. Download the dataset with the Kaggle API

Credentials are looked up in this order:
1. **Colab Secrets** (recommended): `KAGGLE_USERNAME` + `KAGGLE_KEY` (or `KAGGLE_API_TOKEN` if Kaggle gave you a single-string token).
2. An existing `~/.kaggle/kaggle.json`.
3. Otherwise you are prompted to upload `kaggle.json`.

See the README for step-by-step setup.

In [ ]:
import os, pathlib, glob

DATASET  = "lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"
DATA_DIR = pathlib.Path("/content/data")

def configure_kaggle():
    # 1) Colab Secrets
    try:
        from google.colab import userdata
        for name in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
            try:
                value = userdata.get(name)
            except Exception:          # secret missing or notebook access not granted
                value = None
            if value:
                os.environ[name] = value
    except ImportError:
        pass  # not running in Colab
    if (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")) or os.environ.get("KAGGLE_API_TOKEN"):
        print("Kaggle credentials loaded from Colab Secrets / environment.")
        return

    # 2) Existing kaggle.json
    kaggle_json = pathlib.Path.home() / ".kaggle" / "kaggle.json"
    if kaggle_json.exists():
        print("Using existing", kaggle_json)
        return

    # 3) Upload kaggle.json
    from google.colab import files
    print("No secrets found. Please upload your kaggle.json")
    uploaded = files.upload()
    if "kaggle.json" not in uploaded:
        raise RuntimeError("kaggle.json was not uploaded.")
    kaggle_json.parent.mkdir(parents=True, exist_ok=True)
    kaggle_json.write_bytes(uploaded["kaggle.json"])
    kaggle_json.chmod(0o600)   # the kaggle CLI refuses world-readable keys
    print("Saved", kaggle_json)

csv_files = glob.glob(str(DATA_DIR / "*.csv"))
if not csv_files:
    configure_kaggle()
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    subprocess.run(["kaggle", "datasets", "download", "-d", DATASET, "-p", str(DATA_DIR), "--unzip"], check=True)
    csv_files = glob.glob(str(DATA_DIR / "*.csv"))

CSV_PATH = csv_files[0]
print("Dataset:", CSV_PATH)

## 3. Configuration

All hyperparameters live here. `num_layers` is per side (encoder and decoder each get this many LSTM layers).

In [ ]:
from dataclasses import dataclass, asdict

@dataclass
class Config:
    seed: int = 42
    # vocabulary
    max_vocab: int = 20_000        # incl. 4 special tokens
    min_freq: int = 3
    # windows
    src_len: int = 20              # context words fed to the encoder
    tgt_len: int = 5               # words the decoder predicts per window
    stride: int = 4                # step between window starts
    # model
    emb_dim: int = 256
    hidden_dim: int = 256
    num_layers: int = 12           # per side -> 12 encoder + 12 decoder layers
    dropout: float = 0.2
    tie_weights: bool = True       # share embedding and output matrix (needs emb_dim == hidden_dim)
    # training
    batch_size: int = 256
    lr: float = 1e-3
    weight_decay: float = 0.01
    epochs: int = 8
    windows_per_epoch: int = 800_000   # random subset of train windows per epoch (None = all)
    eval_windows: int = 100_000        # fixed subset for val/test
    clip_norm: float = 1.0
    patience: int = 2                  # early stopping on val loss
    use_amp: bool = True               # mixed precision on GPU
    save_to_drive: bool = False        # True -> checkpoint survives Colab disconnects
    ckpt_name: str = "lstm_seq2seq_luong.pt"

cfg = Config()
assert not cfg.tie_weights or cfg.emb_dim == cfg.hidden_dim
cfg

In [ ]:
import random, numpy as np

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = cfg.use_amp and device.type == "cuda"

CKPT_DIR = pathlib.Path("/content/checkpoints")
if cfg.save_to_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT_DIR = pathlib.Path("/content/drive/MyDrive/lstm_next_word")
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = CKPT_DIR / cfg.ckpt_name
print(device, "| AMP:", use_amp, "| checkpoint:", CKPT_PATH)

## 4. Load, clean, deduplicate, split

* Drop the `sentiment` column — only the review text is used.
* The raw text contains HTML (`<br /><br />`, `&amp;`), removed before tokenizing.
* The dataset contains duplicate reviews. They are dropped **before** splitting, otherwise the same review can land in train and test and inflate test scores.
* Split is **by review** (90/5/5), so no window from a test review is ever seen in training.

In [ ]:
import pandas as pd, re, html

df = pd.read_csv(CSV_PATH)
print("raw shape:", df.shape, "| columns:", list(df.columns))
df = df.drop(columns=["sentiment"])
n_before = len(df)
df = df.drop_duplicates(subset="review").reset_index(drop=True)
print(f"dropped {n_before - len(df)} duplicate reviews -> {len(df)} remain")

BR_RE    = re.compile(r"<br\s*/?>", re.IGNORECASE)
TAG_RE   = re.compile(r"<[^>]+>")
TOKEN_RE = re.compile(r"""[a-z0-9]+(?:'[a-z]+)?|[.,!?;:()"-]""")   # words (incl. don't, it's) + punctuation

def clean(text: str) -> str:
    text = html.unescape(text)
    text = BR_RE.sub(" ", text)
    text = TAG_RE.sub(" ", text)
    return text.lower()

def tokenize(text: str) -> list[str]:
    return TOKEN_RE.findall(clean(text))

print(tokenize(df.review[0])[:40])

In [ ]:
from tqdm.auto import tqdm

tokenized = [tokenize(t) for t in tqdm(df.review, desc="tokenizing")]
lengths = np.array([len(t) for t in tokenized])
print(f"tokens/review: mean {lengths.mean():.0f}, median {np.median(lengths):.0f}, max {lengths.max()}")

rng = np.random.default_rng(cfg.seed)
perm = rng.permutation(len(tokenized))
n_hold = int(0.05 * len(perm))
test_ids, val_ids, train_ids = perm[:n_hold], perm[n_hold:2 * n_hold], perm[2 * n_hold:]
print(f"reviews -> train {len(train_ids)}, val {len(val_ids)}, test {len(test_ids)}")

## 5. Vocabulary

Built from the **training split only** (no leakage of val/test words). Special tokens:

| id | token | role |
|---|---|---|
| 0 | `<pad>` | left-padding for short contexts; masked out of attention |
| 1 | `<unk>` | any word outside the vocabulary |
| 2 | `<start>` | start of a review; also the decoder's first input |
| 3 | `<end>` | end of a review (the model learns to predict it) |

In [ ]:
from collections import Counter

PAD, UNK, START, END = "<pad>", "<unk>", "<start>", "<end>"
SPECIALS = [PAD, UNK, START, END]
PAD_ID, UNK_ID, START_ID, END_ID = range(4)

counter = Counter(tok for i in train_ids for tok in tokenized[i])
words = [w for w, c in counter.most_common(cfg.max_vocab - len(SPECIALS)) if c >= cfg.min_freq]
itos = SPECIALS + words
stoi = {w: i for i, w in enumerate(itos)}
V = len(itos)

covered = sum(counter[w] for w in words) / sum(counter.values())
print(f"vocab size {V} | distinct train words {len(counter)} | train token coverage {covered:.2%}")

## 6. Sliding-window samples

Each review becomes `<start> w1 … wn <end>`, left-padded with `src_len-1` `<pad>` tokens so the model also learns from short contexts (e.g. just `<start>`).

A window of `src_len + tgt_len` tokens gives:
* **encoder input** = first `src_len` tokens (the context)
* **decoder target** = next `tgt_len` tokens
* **decoder input** = `<start>` + target shifted right (teacher forcing)

With teacher forcing, decoder step *t* sees the context plus the true words before it, so **every target position is a genuine next-word prediction**.

Windows never cross review boundaries. Reviews are stored in one flat `int32` array; a batch is one vectorized gather — much faster than a per-item `Dataset`.

In [ ]:
WIN = cfg.src_len + cfg.tgt_len
OFFSETS = np.arange(WIN)

def build_split(review_ids):
    pieces, starts, offset = [], [], 0
    for i in review_ids:
        ids = [START_ID] + [stoi.get(t, UNK_ID) for t in tokenized[i]] + [END_ID]
        seq = np.asarray([PAD_ID] * (cfg.src_len - 1) + ids, dtype=np.int32)
        n_windows = len(seq) - WIN + 1
        if n_windows > 0:
            starts.append(offset + np.arange(0, n_windows, cfg.stride))
        pieces.append(seq)
        offset += len(seq)
    return np.concatenate(pieces), np.concatenate(starts)

train_flat, train_starts = build_split(train_ids)
val_flat,   val_starts   = build_split(val_ids)
test_flat,  test_starts  = build_split(test_ids)

def fixed_subset(starts, n, seed):
    if n is None or len(starts) <= n:
        return starts
    return np.sort(np.random.default_rng(seed).choice(starts, n, replace=False))

val_eval_starts  = fixed_subset(val_starts,  cfg.eval_windows, cfg.seed + 1)
test_eval_starts = fixed_subset(test_starts, cfg.eval_windows, cfg.seed + 2)

print(f"windows -> train {len(train_starts):,} | val {len(val_starts):,} (eval on {len(val_eval_starts):,}) "
      f"| test {len(test_starts):,} (eval on {len(test_eval_starts):,})")

def make_batch(flat, starts):
    win = torch.from_numpy(flat[starts[:, None] + OFFSETS].astype(np.int64))
    src = win[:, :cfg.src_len]
    tgt = win[:, cfg.src_len:]
    dec_in = torch.cat([torch.full((len(starts), 1), START_ID, dtype=torch.long), tgt[:, :-1]], dim=1)
    return (src.to(device, non_blocking=True),
            dec_in.to(device, non_blocking=True),
            tgt.to(device, non_blocking=True))

s, d, t = make_batch(train_flat, train_starts[:1])
print("src   :", " ".join(itos[i] for i in s[0].tolist()))
print("dec_in:", " ".join(itos[i] for i in d[0].tolist()))
print("target:", " ".join(itos[i] for i in t[0].tolist()))

## 7. Model

**Why residual connections?** Plain stacks of 12+ LSTM layers train poorly: gradients must pass through every layer's gates. Following GNMT (Wu et al., 2016), each layer from the 2nd onward adds its input to its output (`x = x + LSTM(x)`), giving gradients a shortcut. A LayerNorm at the top of each stack keeps the summed activations at a stable scale before attention.

**Luong attention (general score)**, Luong et al. 2015:
* score(h_t, h̄_s) = h_tᵀ W_a h̄_s
* a_t = softmax over source positions (padding masked out)
* c_t = Σ a_ts h̄_s
* h̃_t = tanh(W_c [c_t ; h_t]) → output layer

The decoder is initialised with the encoder's final (h, c) for each of the 12 layers. Left-padding means the encoder's final state always sits on the last real word, so no sequence packing is needed.

**Not included: Luong's "input feeding"** (feeding h̃_{t-1} into the next decoder step). It forces a Python loop over time steps and loses the cuDNN fused kernel — a large slowdown with 12 layers. Global attention without input feeding is a valid Luong variant.

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F

class ResidualLSTMStack(nn.Module):
    """N single-layer LSTMs with dropout between layers and residual connections from layer 2 onward."""
    def __init__(self, input_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.LSTM(input_dim if i == 0 else hidden_dim, hidden_dim, batch_first=True)
            for i in range(num_layers)
        ])
        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(hidden_dim)
        for lstm in self.layers:   # forget-gate bias = 1 (0.5 + 0.5) -> remembers by default early in training
            for name, p in lstm.named_parameters():
                if name.startswith("bias"):
                    nn.init.zeros_(p)
                    p.data[hidden_dim:2 * hidden_dim].fill_(0.5)   # gate order: input, forget, cell, output

    def forward(self, x, state=None):
        hs, cs = [], []
        for i, lstm in enumerate(self.layers):
            init = None if state is None else (state[0][i:i + 1].contiguous(), state[1][i:i + 1].contiguous())
            out, (h, c) = lstm(x, init)
            out = self.dropout(out)
            x = out if i == 0 else x + out
            hs.append(h); cs.append(c)
        return self.norm(x), (torch.cat(hs, 0), torch.cat(cs, 0))


class LuongAttention(nn.Module):
    """Luong 'general' attention."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.W_a = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.W_c = nn.Linear(2 * hidden_dim, hidden_dim, bias=False)

    def forward(self, dec_out, enc_out, enc_mask):
        # dec_out (B,T,H) | enc_out (B,S,H) | enc_mask (B,S) True = real token
        scores = torch.bmm(dec_out, self.W_a(enc_out).transpose(1, 2)).float()     # (B,T,S), fp32 for a stable softmax
        scores = scores.masked_fill(~enc_mask[:, None, :], float("-inf"))
        attn = torch.softmax(scores, dim=-1)
        context = torch.bmm(attn.to(enc_out.dtype), enc_out)                      # (B,T,H)
        attentional = torch.tanh(self.W_c(torch.cat([context, dec_out], dim=-1)))
        return attentional, attn


class Seq2SeqNextWord(nn.Module):
    def __init__(self, vocab_size, cfg):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, cfg.emb_dim, padding_idx=PAD_ID)  # shared by encoder & decoder
        self.emb_dropout = nn.Dropout(cfg.dropout)
        self.encoder = ResidualLSTMStack(cfg.emb_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout)
        self.decoder = ResidualLSTMStack(cfg.emb_dim, cfg.hidden_dim, cfg.num_layers, cfg.dropout)
        self.attention = LuongAttention(cfg.hidden_dim)
        self.out_dropout = nn.Dropout(cfg.dropout)
        self.proj = nn.Linear(cfg.hidden_dim, vocab_size)
        if cfg.tie_weights:
            self.proj.weight = self.embedding.weight

    def encode(self, src):
        mask = src != PAD_ID
        enc_out, state = self.encoder(self.emb_dropout(self.embedding(src)))
        return enc_out, state, mask

    def decode(self, dec_in, enc_out, mask, state):
        dec_out, state = self.decoder(self.emb_dropout(self.embedding(dec_in)), state)
        attentional, attn = self.attention(dec_out, enc_out, mask)
        return self.proj(self.out_dropout(attentional)), state, attn

    def forward(self, src, dec_in):
        enc_out, state, mask = self.encode(src)
        logits, _, _ = self.decode(dec_in, enc_out, mask, state)
        return logits

model = Seq2SeqNextWord(V, cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"parameters: {n_params / 1e6:.1f}M | layers: {cfg.num_layers} encoder + {cfg.num_layers} decoder")

## 8. Baseline + evaluation

A model is only meaningful relative to a baseline. The **unigram baseline** predicts every word with its training frequency (ignores context). The LSTM must clearly beat its perplexity and accuracy.

Metrics:
* **loss / perplexity** over all target positions
* **top-1 / top-5 accuracy** of the next word

Note: `<unk>` counts as a normal target, so accuracy is slightly optimistic for rare words.

In [ ]:
counts = np.bincount(train_flat[train_flat != PAD_ID], minlength=V).astype(np.float64) + 1.0   # add-one smoothing
uni_logp = np.log(counts / counts.sum())
val_targets = val_flat[val_eval_starts[:, None] + np.arange(cfg.src_len, WIN)]
uni_ppl = math.exp(-uni_logp[val_targets].mean())
top5_uni = np.argsort(-counts)[:5]
uni_top1 = (val_targets == top5_uni[0]).mean()
uni_top5 = np.isin(val_targets, top5_uni).mean()
print(f"Unigram baseline (val): ppl {uni_ppl:.1f} | top-1 {uni_top1:.2%} | top-5 {uni_top5:.2%}")
print("most frequent tokens:", [itos[i] for i in top5_uni])

In [ ]:
@torch.no_grad()
def evaluate(model, flat, starts, batch_size=None):
    model.eval()
    batch_size = batch_size or cfg.batch_size * 2
    tot_loss = tot_tok = top1 = top5 = 0
    for b in range(0, len(starts), batch_size):
        src, dec_in, tgt = make_batch(flat, starts[b:b + batch_size])
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(src, dec_in)
        logits = logits.float().reshape(-1, V)
        y = tgt.reshape(-1)
        tot_loss += F.cross_entropy(logits, y, reduction="sum").item()
        tot_tok += y.numel()
        pred5 = logits.topk(5, dim=-1).indices
        top1 += (pred5[:, 0] == y).sum().item()
        top5 += (pred5 == y[:, None]).any(dim=1).sum().item()
    loss = tot_loss / tot_tok
    return {"loss": loss, "ppl": math.exp(loss), "top1": top1 / tot_tok, "top5": top5 / tot_tok}

## 9. Training

AdamW (no weight decay on biases/LayerNorm), gradient clipping at 1.0 (LSTMs are prone to exploding gradients), mixed precision, LR halved when val loss stops improving, early stopping. The best checkpoint (by val loss) is saved together with the vocabulary and config.

Watch the time of the first few hundred steps and adjust `windows_per_epoch` / `epochs` to your Colab budget.

In [ ]:
import time

decay, no_decay = [], []
for name, p in model.named_parameters():
    (no_decay if p.ndim < 2 else decay).append(p)
optimizer = torch.optim.AdamW([{"params": decay, "weight_decay": cfg.weight_decay},
                               {"params": no_decay, "weight_decay": 0.0}], lr=cfg.lr)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=0)
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

def save_checkpoint(path, epoch, val_metrics):
    torch.save({"model": model.state_dict(), "config": asdict(cfg), "itos": itos,
                "epoch": epoch, "val": val_metrics}, path)

history, best_val, bad_epochs = [], float("inf"), 0
train_rng = np.random.default_rng(cfg.seed)

for epoch in range(1, cfg.epochs + 1):
    model.train()
    order = train_starts[train_rng.permutation(len(train_starts))]
    if cfg.windows_per_epoch:
        order = order[:cfg.windows_per_epoch]
    run_loss = run_tok = 0
    t0 = time.time()
    pbar = tqdm(range(0, len(order), cfg.batch_size), desc=f"epoch {epoch}")
    for b in pbar:
        src, dec_in, tgt = make_batch(train_flat, order[b:b + cfg.batch_size])
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(src, dec_in)
        loss = F.cross_entropy(logits.float().reshape(-1, V), tgt.reshape(-1))
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), cfg.clip_norm)
        scaler.step(optimizer)
        scaler.update()
        run_loss += loss.item() * tgt.numel(); run_tok += tgt.numel()
        if (b // cfg.batch_size) % 50 == 0:
            pbar.set_postfix(loss=f"{run_loss / run_tok:.3f}", lr=f"{optimizer.param_groups[0]['lr']:.1e}")

    train_loss = run_loss / run_tok
    val = evaluate(model, val_flat, val_eval_starts)
    scheduler.step(val["loss"])
    history.append({"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val.items()},
                    "lr": optimizer.param_groups[0]["lr"], "minutes": (time.time() - t0) / 60})
    print(f"epoch {epoch}: train loss {train_loss:.3f} | val loss {val['loss']:.3f} ppl {val['ppl']:.1f} "
          f"top1 {val['top1']:.2%} top5 {val['top5']:.2%} | {history[-1]['minutes']:.1f} min")

    if val["loss"] < best_val - 1e-3:
        best_val, bad_epochs = val["loss"], 0
        save_checkpoint(CKPT_PATH, epoch, val)
        print("  saved best checkpoint")
    else:
        bad_epochs += 1
        if bad_epochs >= cfg.patience:
            print("  early stopping")
            break

pd.DataFrame(history)

In [ ]:
import matplotlib.pyplot as plt

h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.epoch, h.train_loss, "o-", label="train")
ax[0].plot(h.epoch, h.val_loss, "o-", label="val")
ax[0].axhline(math.log(uni_ppl), ls="--", c="grey", label="unigram baseline")
ax[0].set(xlabel="epoch", ylabel="cross-entropy", title="Loss"); ax[0].legend()
ax[1].plot(h.epoch, h.val_top1, "o-", label="top-1")
ax[1].plot(h.epoch, h.val_top5, "o-", label="top-5")
ax[1].set(xlabel="epoch", ylabel="accuracy", title="Val next-word accuracy"); ax[1].legend()
plt.tight_layout(); plt.show()

## 10. Test set (best checkpoint)

Evaluated once, at the end, on reviews never used for training, vocabulary building or model selection.

In [ ]:
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt["model"])
test = evaluate(model, test_flat, test_eval_starts)
test_targets = test_flat[test_eval_starts[:, None] + np.arange(cfg.src_len, WIN)]
uni_test_ppl = math.exp(-uni_logp[test_targets].mean())
print(f"best epoch {ckpt['epoch']}")
print(f"TEST  model  : ppl {test['ppl']:.1f} | top-1 {test['top1']:.2%} | top-5 {test['top5']:.2%}")
print(f"TEST  unigram: ppl {uni_test_ppl:.1f} | top-1 {(test_targets == top5_uni[0]).mean():.2%} "
      f"| top-5 {np.isin(test_targets, top5_uni).mean():.2%}")

## 11. Predict the next word / generate text

At inference the prompt is treated as the start of a review (`<start>` + words), the last `src_len` tokens are encoded, and one decoder step from `<start>` gives the next-word distribution. For longer generation, the chosen word is appended to the context and the process repeats.

`<pad>`, `<start>` and (by default) `<unk>` are blocked from being predicted.

In [ ]:
def encode_prompt(text):
    return [START_ID] + [stoi.get(t, UNK_ID) for t in tokenize(text)]

@torch.no_grad()
def next_word_logits(context_ids):
    model.eval()
    ctx = context_ids[-cfg.src_len:]
    ctx = [PAD_ID] * (cfg.src_len - len(ctx)) + ctx
    src = torch.tensor([ctx], device=device)
    enc_out, state, mask = model.encode(src)
    logits, _, attn = model.decode(torch.tensor([[START_ID]], device=device), enc_out, mask, state)
    return logits[0, -1].float(), attn[0, -1]

def blocked(logits, allow_unk=False):
    logits = logits.clone()
    logits[[PAD_ID, START_ID]] = float("-inf")
    if not allow_unk:
        logits[UNK_ID] = float("-inf")
    return logits

def predict_next(text, k=5, allow_unk=False):
    logits, _ = next_word_logits(encode_prompt(text))
    probs = torch.softmax(blocked(logits, allow_unk), dim=-1)
    p, i = probs.topk(k)
    return [(itos[j], round(q, 4)) for j, q in zip(i.tolist(), p.tolist())]

def detokenize(tokens):
    text = " ".join(tokens)
    text = re.sub(r" ([.,!?;:)])", r"\1", text)
    return text.replace("( ", "(")

def generate(text, n_words=25, temperature=0.8, top_k=40):
    ids, out = encode_prompt(text), []
    for _ in range(n_words):
        logits, _ = next_word_logits(ids)
        logits = blocked(logits) / temperature
        kth = logits.topk(top_k).values[-1]
        logits[logits < kth] = float("-inf")
        nxt = torch.multinomial(torch.softmax(logits, -1), 1).item()
        if nxt == END_ID:
            out.append(END); break
        ids.append(nxt); out.append(itos[nxt])
    return detokenize(tokenize(text) + out)

for prompt in ["this movie was one of the", "i really did not like the", "the acting was", "if you are a fan of"]:
    print(f"{prompt!r:35} -> {predict_next(prompt)}")
print()
for prompt in ["this movie was", "the worst part of the film"]:
    print(generate(prompt), "\n")

### Inspect attention

Which context words did the decoder attend to for its prediction?

In [ ]:
prompt = "i loved the soundtrack but the acting was"
ids = encode_prompt(prompt)
logits, attn = next_word_logits(ids)
ctx = ids[-cfg.src_len:]
weights = attn[-len(ctx):].cpu().numpy()
plt.figure(figsize=(10, 1.6))
plt.imshow(weights[None, :], aspect="auto", cmap="viridis")
plt.xticks(range(len(ctx)), [itos[i] for i in ctx], rotation=45, ha="right")
plt.yticks([]); plt.title(f"next word: {predict_next(prompt, 1)[0][0]!r}")
plt.colorbar(); plt.show()

## 12. Reload later (new Colab session)

Set `cfg.save_to_drive = True` before training if you want the checkpoint to survive a disconnect. To reload you need the same code cells (model classes, tokenizer) and:

```python
ckpt = torch.load(CKPT_PATH, map_location=device)
cfg = Config(**ckpt["config"]); itos = ckpt["itos"]; stoi = {w: i for i, w in enumerate(itos)}; V = len(itos)
model = Seq2SeqNextWord(V, cfg).to(device); model.load_state_dict(ckpt["model"])
```